Colab is making it easier than ever to integrate powerful Generative AI capabilities into your projects. We are launching public preview for a simple and intuitive Python library (google.colab.ai) to access state-of-the-art language models directly within Colab environments. All users have free access to most popular LLMs, while paid users have access to a wider selection of models. This means users can spend less time on configuration and set up and more time bringing their ideas to life. With just a few lines of code, you can now perform a variety of tasks:
- Generate text
- Translate languages
- Write creative content
- Categorize text

Happy Coding!


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/googlecolab/colabtools/blob/main/notebooks/Getting_started_with_google_colab_ai.ipynb)

In [1]:
# ============================================================
# 🏢 ERP-SAP Colab AI Development Assistant
# يدعم: .NET 10 (C#) + React/TypeScript + Clean Architecture + DDD
# نظام ERP كامل — محاسبة، مبيعات، مخازن، HR، مشتريات، ضرائب
# ============================================================

# -------------------- الجزء 1: التثبيت --------------------
# Core AI & GPU
!pip install -q torch transformers accelerate bitsandbytes

# API Server & Networking
!pip install -q fastapi uvicorn pyngrok nest-asyncio

# UI & RAG
!pip install -q gradio langchain langchain-community langchain-text-splitters chromadb sentence-transformers

# File parsing & utilities
!pip install -q pypdf python-multipart aiofiles pygments gitpython

import os, sys, gc, re, shutil, zipfile, threading, time, asyncio, logging, json
from pathlib import Path
from typing import List, Optional, Dict, Any, Tuple
import nest_asyncio
nest_asyncio.apply()

# -------------------- الجزء 2: تحسين GPU --------------------
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import torch
import torch.nn as nn

def optimize_gpu():
    """تهيئة وتحسين GPU لأفضل أداء."""
    if torch.cuda.is_available():
        torch.backends.cudnn.benchmark = True
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
        gpu_name = torch.cuda.get_device_name(0)
        vram = torch.cuda.get_device_properties(0).total_memory / 1e9
        torch.cuda.empty_cache()
        print(f"✅ GPU محدد: {gpu_name}")
        print(f"📦 VRAM: {vram:.1f} GB")
        # تحديد النموذج المناسب بناءً على VRAM
        if vram < 12:
            print("⚠️ VRAM محدود — سيتم استخدام نموذج 7B مع quantization")
            return "small"
        elif vram < 24:
            print("📦 VRAM متوسط — سيتم استخدام نموذج 14B مع 4-bit quantization")
            return "medium"
        else:
            print("🚀 VRAM كبير — يمكن استخدام نموذج كامل")
            return "large"
    else:
        print("⚠️ GPU غير متاح، سيتم استخدام CPU (بطيء جداً)")
        return "cpu"

gpu_tier = optimize_gpu()

# -------------------- الجزء 3: تحميل النموذج --------------------
from transformers import (
    AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig,
    pipeline, TextStreamer
)

# ── ربط Google Drive لحفظ النموذج وتجنب التحميل المتكرر ─────
# يمكنك تفعيل هذا الخيار لكي لا يتم تحميل الـ 30 جيجابايت في كل مرة تشغيل
PERSIST_TO_DRIVE = os.environ.get("PERSIST_TO_DRIVE", "true").lower() == "true"
DRIVE_CACHE_DIR = "/content/drive/MyDrive/huggingface_cache"

if PERSIST_TO_DRIVE:
    try:
        from google.colab import drive
        print("⏳ محاولة ربط Google Drive لحفظ النموذج بشكل دائم...")
        drive.mount('/content/drive', force_remount=True)
        os.makedirs(DRIVE_CACHE_DIR, exist_ok=True)
        os.environ['HF_HOME'] = DRIVE_CACHE_DIR
        print(f"✅ تم ربط Google Drive بنجاح! مسار الحفظ: {DRIVE_CACHE_DIR}")
    except Exception as e:
        print(f"ℹ️ لم يتم ربط Google Drive (سيتم التحميل محلياً للجلسة الحالية فقط): {e}")

# ── مصادقة HuggingFace (اختياري — لبعض النماذج المغلقة) ──────
HF_TOKEN = os.environ.get("HF_TOKEN", None)

# محاولة قراءة التوكن من Colab Secrets
if HF_TOKEN is None:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
        print("🔑 تم قراءة HF_TOKEN من Colab Secrets")
    except Exception:
        pass

if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN, add_to_git_credential=False)
    print("✅ تم تسجيل الدخول في HuggingFace")
else:
    print("ℹ️ لا يوجد HF_TOKEN — سيتم استخدام النماذج العامة فقط")

# ── اختيار النموذج بناءً على GPU ─────────────────────────────
# Qwen2.5-Coder: متاح عام بدون مصادقة، مثبت على T4 Colab
# Qwen3 dense: يحتاج HF_TOKEN في بعض الأحيان
MODEL_OPTIONS = {
    "large":  "Qwen/Qwen2.5-Coder-32B-Instruct",  # يحتاج VRAM 24GB+
    "medium": "Qwen/Qwen2.5-Coder-14B-Instruct",   # مثالي لـ T4 (16GB)
    "small":  "Qwen/Qwen2.5-Coder-7B-Instruct",    # مثالي لـ T4 مع مهام أخرى
    "cpu":    "Qwen/Qwen2.5-Coder-7B-Instruct",
}

# ── سلسلة Fallback — إذا فشل نموذج يجرب الأصغر ──────────────
MODEL_FALLBACK_CHAIN = [
    "Qwen/Qwen2.5-Coder-14B-Instruct",
    "Qwen/Qwen2.5-Coder-7B-Instruct",
    "Qwen/Qwen3-8B",
    "Qwen/Qwen2.5-7B-Instruct",
]

# يمكن للمستخدم تجاوز الاختيار التلقائي عبر متغير بيئة
MODEL_ID = os.environ.get("ERP_MODEL_ID", MODEL_OPTIONS.get(gpu_tier, MODEL_OPTIONS["medium"]))

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

# ── تحميل النموذج مع fallback تلقائي ─────────────────────────
def load_model_with_fallback(primary_model: str, fallback_chain: list):
    """محاولة تحميل النموذج مع سلسلة fallback عند الفشل."""
    models_to_try = [primary_model] + [m for m in fallback_chain if m != primary_model]

    for model_id in models_to_try:
        try:
            print(f"⏳ جارٍ تحميل {model_id} ...")

            tok = AutoTokenizer.from_pretrained(
                model_id,
                trust_remote_code=True,
                token=HF_TOKEN,
            )

            model_kwargs = {
                "quantization_config": quant_config,
                "device_map": "auto",
                "torch_dtype": torch.bfloat16,
                "trust_remote_code": True,
                "token": HF_TOKEN,
            }

            # Flash Attention 2 — متاح فقط على Ampere+ (A100, L4)
            # Tesla T4 (Turing) لا يدعمه — نتخطاه مباشرة
            gpu_supports_fa2 = False
            if torch.cuda.is_available():
                capability = torch.cuda.get_device_capability(0)
                gpu_supports_fa2 = capability[0] >= 8  # Ampere = 8.0+

            if gpu_supports_fa2:
                try:
                    mdl = AutoModelForCausalLM.from_pretrained(
                        model_id, **model_kwargs,
                        attn_implementation="flash_attention_2"
                    )
                    print("⚡ Flash Attention 2 مفعّل")
                except Exception:
                    mdl = AutoModelForCausalLM.from_pretrained(model_id, **model_kwargs)
                    print("📌 يعمل بدون Flash Attention")
            else:
                mdl = AutoModelForCausalLM.from_pretrained(model_id, **model_kwargs)
                if torch.cuda.is_available():
                    print(f"📌 GPU {torch.cuda.get_device_name(0)} — لا يدعم Flash Attention 2")

            if tok.pad_token is None:
                tok.pad_token = tok.eos_token

            print(f"✅ تم تحميل النموذج {model_id} بنجاح!")
            return tok, mdl, model_id

        except Exception as e:
            error_msg = str(e)
            if "401" in error_msg or "Repository Not Found" in error_msg:
                print(f"🔒 {model_id} — يتطلب مصادقة أو غير متاح. جارٍ تجربة البديل...")
            elif "out of memory" in error_msg.lower() or "CUDA" in error_msg:
                print(f"💾 {model_id} — لا يوجد ذاكرة كافية. جارٍ تجربة نموذج أصغر...")
                torch.cuda.empty_cache()
                gc.collect()
            else:
                print(f"⚠️ {model_id} — خطأ: {error_msg[:200]}")
            continue

    raise RuntimeError(
        "❌ فشل تحميل جميع النماذج!\n"
        "الحلول:\n"
        "1. أضف HF_TOKEN في Colab Secrets (https://huggingface.co/settings/tokens)\n"
        "2. حدد نموذج يدوياً: os.environ['ERP_MODEL_ID'] = 'Qwen/Qwen2.5-Coder-7B-Instruct'\n"
        "3. تأكد من أنك تستخدم GPU runtime في Colab"
    )

tokenizer, model, MODEL_ID = load_model_with_fallback(MODEL_ID, MODEL_FALLBACK_CHAIN)
print(f"🤖 النموذج النشط: {MODEL_ID}")

# -------------------- الجزء 4: نظام RAG لـ ERP-SAP --------------------
try:
    from langchain_text_splitters import RecursiveCharacterTextSplitter
except ImportError:
    from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
try:
    from langchain_core.documents import Document
except ImportError:
    from langchain.schema import Document

# ── Embeddings ────────────────────────────────────────────────
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={'device': 'cuda' if torch.cuda.is_available() else 'cpu'}
)

# ── Vector Store ──────────────────────────────────────────────
vector_store = Chroma(
    collection_name="erp_sap_codebase",
    embedding_function=embedding_model,
    persist_directory="./chroma_erp_db"
)

# ── Text Splitter — محسّن لملفات الكود الكبيرة ──────────────
# فواصل ذكية تراعي بنية C#, TypeScript, CSS
code_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=300,
    separators=[
        # C# separators
        "\nnamespace ", "\npublic class ", "\npublic interface ",
        "\npublic record ", "\npublic enum ", "\npublic static class ",
        "\npublic async ", "\npublic ", "\nprivate ",
        # TypeScript/React separators
        "\nexport default ", "\nexport const ", "\nexport function ",
        "\nexport interface ", "\nexport type ", "\nexport enum ",
        "\nconst ", "\nfunction ",
        # General separators
        "\n\n", "\n", " ", ""
    ]
)

# ── ذاكرة المحادثة ───────────────────────────────────────────
chat_memory = []

# ============================================================
# 🏗️ ERP-SAP Project Analyzer — يفهم بنية المشروع
# ============================================================

# ── خريطة الامتدادات الكاملة للمشروع ─────────────────────────
ERP_FILE_EXTENSIONS = {
    # Backend — .NET 10 / C#
    ".cs":       "C#",
    ".csproj":   "MSBuild/C#",

    # Frontend — React / TypeScript
    ".tsx":      "TypeScript/React",
    ".ts":       "TypeScript",
    ".jsx":      "JavaScript/React",
    ".js":       "JavaScript",
    ".css":      "CSS",

    # Configuration
    ".json":     "JSON",
    ".yaml":     "YAML",
    ".yml":      "YAML",

    # Database & Migrations
    ".sql":      "SQL",

    # Documentation & Architecture
    ".md":       "Markdown",

    # Docker & DevOps
    ".dockerfile": "Docker",

    # HTML templates
    ".html":     "HTML",
}

# ── ملفات يجب تجاهلها ────────────────────────────────────────
IGNORE_PATTERNS = {
    "node_modules", "bin", "obj", "dist", ".git", ".turbo",
    "__generated__", "playwright-report", "test-results",
    ".next", "coverage", ".planning", ".pi", ".agents",
    "pnpm-lock.yaml", "package-lock.json",
}

# ── أسماء ملفات خاصة بدون امتداد ─────────────────────────────
SPECIAL_FILES = {"Dockerfile", "docker-compose.yml", "docker-compose.override.yml"}

# ── خريطة الطبقات المعمارية ──────────────────────────────────
ARCHITECTURE_LAYERS = {
    "Erp.Domain":          "Domain Layer (Entities, Value Objects, Domain Events)",
    "Erp.Application":     "Application Layer (Use Cases, DTOs, Interfaces)",
    "Erp.Infrastructure":  "Infrastructure Layer (EF Core, External Services, Persistence)",
    "Erp.Api":             "API Layer (Controllers, Middleware, Endpoints)",
    "Erp.Ai":              "AI Module (RAG, Agents, Copilot, Skills, MCP)",
    "Erp.Ai.Abstractions": "AI Abstractions (Interfaces, Contracts)",
    "Erp.Ai.Mcp":          "MCP Server (Model Context Protocol)",
}

# ── خريطة وحدات ERP ──────────────────────────────────────────
ERP_MODULES = {
    "Accounting":     "المحاسبة العامة — GL, دفتر اليومية, ميزان المراجعة",
    "Sales":          "المبيعات — عروض أسعار, أوامر بيع, فواتير",
    "Inventory":      "المخازن — أصناف, حركات مخزنية, تقييم",
    "Procurement":    "المشتريات — طلبات شراء, أوامر شراء, استلام",
    "Finance":        "المالية — العملات, أسعار الصرف, الميزانية",
    "Hr":             "الموارد البشرية — موظفين, هيكل تنظيمي",
    "Payroll":        "الرواتب — حساب الرواتب, البدلات, الاستقطاعات",
    "Tax":            "الضرائب — ضريبة القيمة المضافة, الإعفاءات",
    "Identity":       "الهوية — مستخدمين, أدوار, صلاحيات (RBAC)",
    "Security":       "الأمان — تشفير, audit logging, HSTS",
    "Platform":       "المنصة — multi-tenancy, feature flags, إعدادات",
    "Organization":   "التنظيم — شركات, فروع, أقسام",
    "Customers":      "العملاء — بيانات العملاء, حسابات العملاء",
    "Parties":        "الأطراف — أطراف مشتركة (عملاء/موردين)",
    "Pricing":        "التسعير — قوائم أسعار, خصومات, عروض",
    "Subledger":      "دفاتر فرعية — ربط المخزون بـ GL",
    "Localization":   "التعريب — i18n, RTL, ترجمة",
    "Configuration":  "الإعدادات — إعدادات النظام العامة",
    "DocumentNumbering": "الترقيم — تسلسلات المستندات",
    "Integration":    "التكامل — APIs خارجية, webhooks",
    "Enterprise":     "المؤسسة — إعدادات مؤسسية",
    "Reporting":      "التقارير — تقارير مالية ومخزنية",
    "Discounts":      "الخصومات — أنواع الخصومات والعروض",
    "MultiCurrency":  "العملات المتعددة — تحويل عملات, triangulation",
    "Approvals":      "الموافقات — سلسلة الموافقات",
    "System":         "النظام — إعدادات وأدوات النظام",
}

# ── خريطة حزم Frontend ────────────────────────────────────────
FRONTEND_PACKAGES = {
    "design-tokens":   "Design Tokens — ألوان, خطوط, مسافات, ظلال",
    "ui-core":         "UI Core — مكونات أساسية (Button, Input, Modal)",
    "ui-controls":     "UI Controls — مكونات متقدمة (DatePicker, Select)",
    "ui-forms":        "UI Forms — نماذج ديناميكية",
    "ui-grid":         "UI Grid — جداول بيانات متقدمة (AG Grid)",
    "api-client":      "API Client — Axios wrapper + interceptors",
    "auth-context":    "Auth Context — JWT + tenant context",
    "i18n":            "i18n — ترجمة (ar/en) + RTL support",
    "theming":         "Theming — light/dark mode + custom themes",
    "permissions":     "Permissions — RBAC frontend guards",
    "multitenancy":    "Multi-tenancy — tenant switching + isolation",
    "notifications":   "Notifications — toast + push notifications",
    "observability":   "Observability — error tracking + analytics",
    "security":        "Security — XSS protection + CSP",
    "dynamic-forms":   "Dynamic Forms — schema-driven form generation",
    "approvals":       "Approvals — workflow approvals UI",
    "utils":           "Utils — shared utilities",
}


class ErpProjectAnalyzer:
    """محلل مشاريع ERP-SAP — يفهم Clean Architecture و DDD."""

    @staticmethod
    def extract_zip(zip_path: str, extract_to: str = "/content/erp_project") -> str:
        """استخراج ملف ZIP للمشروع."""
        shutil.rmtree(extract_to, ignore_errors=True)
        os.makedirs(extract_to, exist_ok=True)
        with zipfile.ZipFile(zip_path, 'r') as z:
            z.extractall(extract_to)
        return extract_to

    @staticmethod
    def should_ignore(path: str) -> bool:
        """تحقق هل المسار يجب تجاهله."""
        parts = Path(path).parts
        return any(part in IGNORE_PATTERNS for part in parts)

    @staticmethod
    def detect_architecture_layer(file_path: str) -> str:
        """اكتشاف الطبقة المعمارية من مسار الملف."""
        for layer_key, layer_desc in ARCHITECTURE_LAYERS.items():
            if layer_key in file_path:
                return layer_desc
        # Frontend layers
        if "erp-frontend" in file_path:
            if "/apps/" in file_path:
                return "Frontend App (React SPA)"
            if "/packages/" in file_path:
                return "Frontend Shared Package"
        if "docker" in file_path.lower():
            return "DevOps / Containerization"
        if "erp-architecture-registry" in file_path:
            return "Architecture Registry (Governance)"
        if "e2e" in file_path or "tests" in file_path:
            return "Testing"
        return "Other"

    @staticmethod
    def detect_module(file_path: str) -> str:
        """اكتشاف وحدة ERP من مسار الملف."""
        for module_key in ERP_MODULES:
            # Case-insensitive match in path
            if module_key.lower() in file_path.lower():
                return f"{module_key} — {ERP_MODULES[module_key]}"
        # Frontend packages
        for pkg_key in FRONTEND_PACKAGES:
            if pkg_key in file_path:
                return f"{pkg_key} — {FRONTEND_PACKAGES[pkg_key]}"
        return "General"

    @staticmethod
    def detect_component_type(file_path: str, content: str) -> str:
        """اكتشاف نوع المكون البرمجي."""
        fname = os.path.basename(file_path).lower()
        ext = os.path.splitext(file_path)[1]

        # C# patterns
        if ext == ".cs":
            if "Controller" in file_path:
                return "API Controller"
            if "Entity" in fname or "Entities" in fname:
                return "Domain Entity"
            if "Service" in fname:
                return "Application Service"
            if "Repository" in fname:
                return "Repository"
            if "DependencyInjection" in fname:
                return "DI Registration"
            if "Middleware" in fname:
                return "Middleware"
            if "Hub" in fname:
                return "SignalR Hub"
            if "public record " in content:
                return "DTO / Record"
            if "public interface " in content:
                return "Interface / Contract"
            if "public enum " in content:
                return "Enum"
            return "C# Class"

        # TypeScript/React patterns
        if ext in (".tsx", ".ts"):
            if "page" in fname or "Page" in fname:
                return "React Page"
            if "component" in fname.lower() or file_path.endswith(".tsx"):
                if "export default function" in content or "export const" in content:
                    return "React Component"
            if "hook" in fname.lower() or fname.startswith("use"):
                return "React Hook"
            if "store" in fname.lower():
                return "Zustand Store"
            if "context" in fname.lower():
                return "React Context"
            if "type" in fname.lower() or "interface" in content[:500]:
                return "TypeScript Types"
            return "TypeScript Module"

        # Config patterns
        if fname in ("package.json", "tsconfig.json", "turbo.json"):
            return "Build Config"
        if fname in ("appsettings.json", "appsettings.development.json"):
            return "App Configuration"
        if fname.endswith(".csproj"):
            return "Project File"
        if "docker-compose" in fname or fname == "dockerfile":
            return "Docker Config"

        return "Source File"

    @staticmethod
    def load_and_index_project(folder_path: str) -> Dict[str, Any]:
        """تحليل وفهرسة مشروع ERP-SAP بالكامل."""
        docs = []
        stats = {
            "total_files": 0,
            "indexed_files": 0,
            "total_chunks": 0,
            "by_layer": {},
            "by_module": {},
            "by_language": {},
        }

        for root, dirs, files in os.walk(folder_path):
            # تصفية المجلدات المتجاهلة
            dirs[:] = [d for d in dirs if d not in IGNORE_PATTERNS]

            for file in files:
                file_path = os.path.join(root, file)
                stats["total_files"] += 1

                # تحقق من التجاهل
                if ErpProjectAnalyzer.should_ignore(file_path):
                    continue

                # تحقق من الامتداد
                ext = os.path.splitext(file)[1].lower()
                is_special = file in SPECIAL_FILES

                if ext not in ERP_FILE_EXTENSIONS and not is_special:
                    continue

                try:
                    with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
                        content = f.read()

                    if len(content.strip()) < 10:
                        continue

                    # تحديد اللغة
                    language = ERP_FILE_EXTENSIONS.get(ext, "Docker" if is_special else "Unknown")

                    # اكتشاف المعمارية
                    layer = ErpProjectAnalyzer.detect_architecture_layer(file_path)
                    module = ErpProjectAnalyzer.detect_module(file_path)
                    component_type = ErpProjectAnalyzer.detect_component_type(file_path, content)

                    # بناء Metadata غنية
                    meta = {
                        "source": file_path,
                        "filename": file,
                        "language": language,
                        "layer": layer,
                        "module": module,
                        "component_type": component_type,
                    }

                    # تقسيم الملف إلى أجزاء
                    chunks = code_splitter.split_text(content)
                    for i, chunk in enumerate(chunks):
                        chunk_meta = {**meta, "chunk_index": i, "total_chunks": len(chunks)}
                        docs.append(Document(page_content=chunk, metadata=chunk_meta))

                    # إحصائيات
                    stats["indexed_files"] += 1
                    stats["by_layer"][layer] = stats["by_layer"].get(layer, 0) + 1
                    stats["by_module"][module] = stats["by_module"].get(module, 0) + 1
                    stats["by_language"][language] = stats["by_language"].get(language, 0) + 1

                except Exception as e:
                    print(f"⚠️ خطأ في قراءة {file_path}: {e}")

        if docs:
            # إضافة على دفعات لتجنب مشاكل الذاكرة
            batch_size = 100
            for i in range(0, len(docs), batch_size):
                batch = docs[i:i + batch_size]
                vector_store.add_documents(batch)
            vector_store.persist()
            stats["total_chunks"] = len(docs)
            print(f"✅ تم فهرسة {stats['indexed_files']} ملف ({stats['total_chunks']} مقطع)")
        else:
            print("⚠️ لم يتم العثور على ملفات مدعومة")

        return stats

    @staticmethod
    def get_project_summary(folder_path: str) -> str:
        """توليد ملخص هيكلي للمشروع."""
        summary_lines = ["# 🏢 ERP-SAP Project Structure\n"]

        # Backend
        dotnet_path = os.path.join(folder_path, "dotnet", "src")
        if os.path.exists(dotnet_path):
            summary_lines.append("## 🔧 Backend (.NET 10 — Clean Architecture)")
            for item in sorted(os.listdir(dotnet_path)):
                if os.path.isdir(os.path.join(dotnet_path, item)):
                    layer_desc = ARCHITECTURE_LAYERS.get(item, item)
                    summary_lines.append(f"  - `{item}/` → {layer_desc}")

        # Frontend
        frontend_path = os.path.join(folder_path, "erp-frontend")
        if os.path.exists(frontend_path):
            summary_lines.append("\n## 🎨 Frontend (React + TypeScript — Monorepo)")
            apps_path = os.path.join(frontend_path, "apps")
            if os.path.exists(apps_path):
                summary_lines.append("  ### Apps:")
                for app in sorted(os.listdir(apps_path)):
                    if os.path.isdir(os.path.join(apps_path, app)):
                        summary_lines.append(f"    - `{app}/`")
            pkgs_path = os.path.join(frontend_path, "packages")
            if os.path.exists(pkgs_path):
                summary_lines.append("  ### Shared Packages:")
                for pkg in sorted(os.listdir(pkgs_path)):
                    if os.path.isdir(os.path.join(pkgs_path, pkg)):
                        desc = FRONTEND_PACKAGES.get(pkg, pkg)
                        summary_lines.append(f"    - `{pkg}/` → {desc}")

        # Docker
        docker_compose = os.path.join(folder_path, "docker-compose.yml")
        if os.path.exists(docker_compose):
            summary_lines.append("\n## 🐳 Infrastructure")
            summary_lines.append("  - PostgreSQL 16 + Redis 7 + Nginx")
            summary_lines.append("  - Docker Compose (production + dev-watch + observability)")

        return "\n".join(summary_lines)

# -------------------- الجزء 5: OpenAI API متوافق --------------------
from fastapi import FastAPI, HTTPException, UploadFile, File
from fastapi.responses import StreamingResponse, JSONResponse
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel, Field
import uvicorn

app = FastAPI(
    title="ERP-SAP AI Assistant API",
    description="OpenAI-compatible API for ERP-SAP development",
    version="2.0.0"
)

# CORS — السماح بالوصول من أي مصدر (للتطوير)
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

class ChatMessage(BaseModel):
    role: str
    content: str

class ChatRequest(BaseModel):
    model: str = MODEL_ID
    messages: List[ChatMessage]
    stream: bool = False
    temperature: float = 0.6
    max_tokens: int = 4096
    top_p: float = 0.9

# ── System Prompt متخصص بـ ERP-SAP ──────────────────────────
ERP_SYSTEM_PROMPT = """أنت مساعد ذكي متخصص في تطوير نظام ERP-SAP — نظام تخطيط موارد المؤسسات.

## 🏗️ البنية المعمارية (Clean Architecture + DDD)
- **Domain Layer** (`Erp.Domain`): الكيانات, Value Objects, Domain Events, Enums
- **Application Layer** (`Erp.Application`): Use Cases, DTOs, Interfaces, CQRS pattern
- **Infrastructure Layer** (`Erp.Infrastructure`): EF Core, PostgreSQL, Redis, External Services
- **API Layer** (`Erp.Api`): RESTful Controllers V1, Middleware, SignalR Hubs, JWT Auth
- **AI Module** (`Erp.Ai`): RAG Pipeline, Agents, Copilot, Skills, MCP Server, Guardrails

## 🖥️ التقنيات
- **Backend**: .NET 10, C# 13, EF Core, PostgreSQL 16, Redis 7
- **Frontend**: React 19, TypeScript, Vite, Zustand, pnpm monorepo, Turborepo
- **Testing**: Playwright (E2E), Vitest (unit), Storybook
- **DevOps**: Docker Compose, Nginx, OpenTelemetry, Grafana
- **Design System**: Design Tokens (CSS custom properties), RTL/LTR, dark/light themes

## 📦 الوحدات (27 وحدة)
المحاسبة, المبيعات, المخازن, المشتريات, المالية, الموارد البشرية, الرواتب,
الضرائب, الهوية/RBAC, الأمان, المنصة, التنظيم, العملاء, الأطراف,
التسعير, الخصومات, دفاتر فرعية, التعريب (i18n), الإعدادات, ترقيم المستندات,
التكامل, المؤسسة, التقارير, العملات المتعددة, الموافقات, النظام

## 🔒 المبادئ الملزمة
- Multi-tenancy: عزل كامل بين المستأجرين (tenant isolation via TenantId)
- Audit Logging: كل تغيير يُسجّل (CreatedBy, ModifiedBy, timestamps)
- RBAC: صلاحيات دقيقة لكل endpoint
- Soft Deletes: لا حذف فعلي — IsDeleted flag
- Precision: 4 أرقام عشرية للمبالغ, 6 لأسعار الصرف
- API Versioning: /api/v1/* — كل controllers في مجلد V1

## 📐 قواعد الكود
- Backend: Clean Code, SOLID, DRY, Repository pattern, DI everywhere
- Frontend: Composition pattern, custom hooks, Zustand stores, CSS modules
- التوثيق: XML docs على كل public API, JSDoc على components
- الأمان: لا hardcoded secrets, CSP headers, XSS protection

استخدم سياق RAG المرفق إذا كان متاحاً للإجابة بدقة أكبر.
قدم أمثلة عملية من بنية المشروع الفعلية عند الإمكان.
"""


def build_prompt(messages: List[ChatMessage], system_prompt: Optional[str] = None) -> str:
    """بناء prompt بتنسيق Qwen3-Coder."""
    if system_prompt is None:
        system_prompt = ERP_SYSTEM_PROMPT

    formatted = f"<|im_start|>system\n{system_prompt}<|im_end|>\n"
    for msg in messages:
        if msg.role == "user":
            formatted += f"<|im_start|>user\n{msg.content}<|im_end|>\n"
        elif msg.role == "assistant":
            formatted += f"<|im_start|>assistant\n{msg.content}<|im_end|>\n"
    formatted += "<|im_start|>assistant\n"
    return formatted


def detect_query_domain(query: str) -> str:
    """اكتشاف مجال السؤال لتحسين البحث في RAG."""
    query_lower = query.lower()

    backend_keywords = ["controller", "service", "entity", "repository", "c#", "csharp",
                        ".net", "api", "endpoint", "middleware", "ef core", "migration",
                        "domain", "infrastructure", "ddd", "cqrs"]
    frontend_keywords = ["component", "react", "typescript", "tsx", "hook", "zustand",
                         "store", "css", "design token", "theme", "i18n", "rtl",
                         "vite", "frontend", "ui"]
    module_keywords = {
        "accounting": ["محاسبة", "journal", "gl", "ledger", "قيد", "يومية"],
        "sales": ["مبيعات", "sales", "invoice", "فاتورة", "quotation", "عرض سعر"],
        "inventory": ["مخازن", "inventory", "stock", "مخزون", "item", "صنف"],
        "hr": ["hr", "موارد بشرية", "employee", "موظف"],
        "procurement": ["مشتريات", "purchase", "procurement", "أمر شراء"],
    }

    if any(kw in query_lower for kw in backend_keywords):
        return "backend"
    if any(kw in query_lower for kw in frontend_keywords):
        return "frontend"
    for module, keywords in module_keywords.items():
        if any(kw in query_lower for kw in keywords):
            return module
    return "general"


def generate_response(prompt: str, temperature: float = 0.6,
                      max_tokens: int = 4096, stream: bool = False):
    """توليد رد من النموذج مع تحسينات."""
    # Qwen3-Coder يدعم context window أكبر
    max_input_length = 32768 if gpu_tier in ("large", "medium") else 16384

    inputs = tokenizer(
        prompt, return_tensors="pt",
        truncation=True, max_length=max_input_length
    ).to(model.device)

    if stream:
        from transformers import TextIteratorStreamer
        streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, timeout=120)
        generation_kwargs = dict(
            inputs, streamer=streamer,
            max_new_tokens=max_tokens,
            temperature=max(temperature, 0.01),  # تجنب temperature=0 في بعض النماذج
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.05,  # تقليل التكرار
        )
        thread = threading.Thread(target=model.generate, kwargs=generation_kwargs)
        thread.start()
        return streamer

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            temperature=max(temperature, 0.01),
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.05,
        )
    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

    # تنظيف الذاكرة GPU
    del inputs, outputs
    torch.cuda.empty_cache()

    return response

# ── نقاط النهاية (Endpoints) ──────────────────────────────────

@app.get("/health")
async def health_check():
    """فحص صحة الخادم."""
    return {
        "status": "healthy",
        "model": MODEL_ID,
        "gpu": torch.cuda.is_available(),
        "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
        "indexed_chunks": vector_store._collection.count(),
    }


@app.get("/v1/models")
async def list_models():
    """قائمة النماذج المتاحة — متوافق مع OpenAI API."""
    return {
        "object": "list",
        "data": [{
            "id": MODEL_ID,
            "object": "model",
            "owned_by": "local",
            "permission": []
        }]
    }


@app.post("/v1/chat/completions")
async def chat_completion(request: ChatRequest):
    """نقطة المحادثة الرئيسية — متوافق مع OpenAI API."""
    try:
        # 1. استخراج آخر رسالة من المستخدم
        last_user_msg = next(
            (m.content for m in reversed(request.messages) if m.role == "user"), ""
        )

        # 2. اكتشاف مجال السؤال
        domain = detect_query_domain(last_user_msg)

        # 3. استرجاع سياق RAG مع تصفية حسب المجال
        context = ""
        if len(last_user_msg) > 10:
            search_kwargs = {"k": 5}

            # تصفية حسب المجال إذا أمكن
            if domain == "backend":
                search_kwargs["filter"] = {"language": {"$in": ["C#", "MSBuild/C#"]}}
            elif domain == "frontend":
                search_kwargs["filter"] = {"language": {"$in": ["TypeScript/React", "TypeScript", "CSS"]}}

            try:
                docs = vector_store.similarity_search(last_user_msg, **search_kwargs)
            except Exception:
                # fallback بدون filter
                docs = vector_store.similarity_search(last_user_msg, k=5)

            if docs:
                context_parts = []
                for d in docs:
                    meta = d.metadata
                    header = f"📁 {meta.get('filename', '?')} | {meta.get('layer', '?')} | {meta.get('component_type', '?')}"
                    context_parts.append(f"--- {header} ---\n{d.page_content}")
                context = "\n\n".join(context_parts)

        # 4. بناء الرسائل مع السياق
        if context:
            rag_prefix = f"📚 سياق من المشروع (RAG — {domain}):\n{context}\n\n---\nالسؤال: "
            modified_messages = []
            for m in request.messages:
                if m.role == "user" and m.content == last_user_msg:
                    modified_messages.append(
                        ChatMessage(role="user", content=rag_prefix + m.content)
                    )
                else:
                    modified_messages.append(m)
        else:
            modified_messages = request.messages

        # 5. إضافة ذاكرة الجلسة (آخر 8 رسائل)
        global chat_memory
        memory_context = "\n".join(chat_memory[-8:])
        sys_prompt = ERP_SYSTEM_PROMPT
        if memory_context:
            sys_prompt += f"\n\n## ملخص المحادثة السابقة:\n{memory_context}"

        prompt = build_prompt(modified_messages, sys_prompt)

        # 6. توليد الرد
        if request.stream:
            async def stream_generator():
                streamer = generate_response(
                    prompt, request.temperature, request.max_tokens, stream=True
                )
                for chunk in streamer:
                    data = json.dumps({
                        "id": f"chatcmpl-erp-{int(time.time())}",
                        "object": "chat.completion.chunk",
                        "created": int(time.time()),
                        "model": MODEL_ID,
                        "choices": [{"index": 0, "delta": {"content": chunk}, "finish_reason": None}]
                    })
                    yield f"data: {data}\n\n"
                yield "data: [DONE]\n\n"
            return StreamingResponse(stream_generator(), media_type="text/event-stream")
        else:
            response_text = generate_response(
                prompt, request.temperature, request.max_tokens, stream=False
            )

            # حفظ في الذاكرة
            chat_memory.append(f"User: {last_user_msg[:150]}")
            chat_memory.append(f"AI: {response_text[:150]}")
            if len(chat_memory) > 30:
                chat_memory = chat_memory[-20:]

            return {
                "id": f"chatcmpl-erp-{int(time.time())}",
                "object": "chat.completion",
                "created": int(time.time()),
                "model": MODEL_ID,
                "choices": [{
                    "index": 0,
                    "message": {"role": "assistant", "content": response_text},
                    "finish_reason": "stop"
                }],
                "usage": {
                    "prompt_tokens": len(tokenizer.encode(prompt)),
                    "completion_tokens": len(tokenizer.encode(response_text)),
                    "total_tokens": len(tokenizer.encode(prompt)) + len(tokenizer.encode(response_text))
                }
            }
    except Exception as e:
        import traceback
        traceback.print_exc()
        raise HTTPException(status_code=500, detail=str(e))


@app.post("/upload_project")
async def upload_project(file: UploadFile = File(...)):
    """رفع وتحليل مشروع ERP-SAP (ZIP)."""
    try:
        temp_zip = f"/tmp/{file.filename}"
        with open(temp_zip, "wb") as f:
            f.write(await file.read())

        extract_path = ErpProjectAnalyzer.extract_zip(temp_zip)
        stats = ErpProjectAnalyzer.load_and_index_project(extract_path)
        summary = ErpProjectAnalyzer.get_project_summary(extract_path)

        # تنظيف
        os.remove(temp_zip)
        shutil.rmtree(extract_path, ignore_errors=True)

        return {
            "status": "success",
            "stats": stats,
            "summary": summary,
            "message": f"✅ تم تحليل {stats['indexed_files']} ملف ({stats['total_chunks']} مقطع)"
        }
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))


@app.get("/project/stats")
async def project_stats():
    """إحصائيات المشروع المفهرس."""
    count = vector_store._collection.count()
    return {
        "indexed_chunks": count,
        "model": MODEL_ID,
        "gpu_tier": gpu_tier,
    }


# -------------------- الجزء 6: واجهة Gradio --------------------
import gradio as gr

def gradio_chat(message: str, history: list) -> str:
    """دالة المحادثة لواجهة Gradio."""
    from fastapi.testclient import TestClient
    client = TestClient(app)

    # بناء الرسائل من التاريخ بشكل مرن يتوافق مع مختلف إصدارات Gradio
    messages = []
    for h in history:
        if isinstance(h, dict):
            role = h.get("role")
            content = h.get("content")
            if role and content:
                messages.append({"role": role, "content": content})
        elif hasattr(h, "role") and hasattr(h, "content"):
            messages.append({"role": h.role, "content": h.content})
        elif isinstance(h, (list, tuple)) and len(h) >= 2:
            if h[0]:
                messages.append({"role": "user", "content": h[0]})
            if h[1]:
                messages.append({"role": "assistant", "content": h[1]})
    messages.append({"role": "user", "content": message})

    response = client.post("/v1/chat/completions", json={
        "model": MODEL_ID,
        "messages": messages,
        "stream": False,
        "temperature": 0.6,
        "max_tokens": 4096
    })

    if response.status_code == 200:
        return response.json()["choices"][0]["message"]["content"]
    else:
        return f"❌ خطأ: {response.text}"


# ── واجهة Gradio المحسّنة ─────────────────────────────────────
with gr.Blocks(
    theme=gr.themes.Soft(
        primary_hue="blue",
        secondary_hue="slate",
    ),
    title="ERP-SAP AI Assistant",
    css="""
    .gradio-container { max-width: 1400px !important; }
    .erp-header { text-align: center; padding: 1rem; }
    .stats-box { background: #1e293b; color: #e2e8f0; padding: 1rem;
                 border-radius: 8px; font-family: monospace; font-size: 0.85rem; }
    """
) as demo:

    gr.Markdown("""
    <div class="erp-header">

    # 🏢 ERP-SAP AI Development Assistant
    ### مساعد تطوير نظام تخطيط موارد المؤسسات

    **Backend:** .NET 10 / C# 13 / Clean Architecture &nbsp;|&nbsp;
    **Frontend:** React 19 / TypeScript / Vite &nbsp;|&nbsp;
    **AI Model:** """ + MODEL_ID + """

    </div>
    """)

    with gr.Row():
        # ── العمود الرئيسي — المحادثة ──────────────────────
        with gr.Column(scale=3):
            chatbot = gr.Chatbot(
                height=600,
                label="💬 المساعد الذكي — ERP-SAP",
            )
            with gr.Row():
                msg = gr.Textbox(
                    placeholder="اسأل عن الأكواد، البنية المعمارية، أو اقتراح تحسينات...",
                    label="✏️ الإدخال",
                    scale=5,
                    lines=2,
                )
                send_btn = gr.Button("إرسال ➤", variant="primary", scale=1)
            clear = gr.Button("🧹 مسح المحادثة", variant="secondary")

        # ── العمود الجانبي — الأدوات ──────────────────────
        with gr.Column(scale=1):
            gr.Markdown("### 📁 رفع المشروع")
            file_upload = gr.File(
                label="رفع مشروع (ZIP)",
                file_types=[".zip"],
            )
            upload_status = gr.Textbox(label="حالة التحليل", interactive=False, lines=3)

            gr.Markdown("### 📊 إحصائيات")
            stats_display = gr.Textbox(
                label="معلومات",
                interactive=False,
                lines=4,
                elem_classes=["stats-box"],
            )
            btn_stats = gr.Button("🔄 تحديث الإحصائيات")

            gr.Markdown("### ⚙️ أدوات")
            btn_clear_rag = gr.Button("🗑️ مسح ذاكرة RAG")

            gr.Markdown("### 📚 الوحدات المدعومة")
            gr.Markdown("""
            `Accounting` `Sales` `Inventory`
            `Procurement` `Finance` `HR`
            `Payroll` `Tax` `Identity`
            `Security` `Platform` `Pricing`
            `+15 وحدة أخرى`
            """)

            gr.Markdown("### 🗂️ الملفات المدعومة")
            gr.Markdown("""
            `.cs` `.tsx` `.ts` `.css`
            `.json` `.yaml` `.sql` `.md`
            `.csproj` `Dockerfile`
            """)

    # ── الأحداث ───────────────────────────────────────────

    def respond(message, history):
        if not message.strip():
            return "", history
        bot_reply = gradio_chat(message, history)
        return "", history + [[message, bot_reply]]

    def upload_project_handler(file):
        if file is None:
            return "❌ الرجاء اختيار ملف ZIP"
        try:
            extract_path = ErpProjectAnalyzer.extract_zip(file.name)
            stats = ErpProjectAnalyzer.load_and_index_project(extract_path)
            shutil.rmtree(extract_path, ignore_errors=True)

            result = f"✅ تم تحليل المشروع!\n"
            result += f"📄 ملفات مفهرسة: {stats['indexed_files']}\n"
            result += f"📦 مقاطع: {stats['total_chunks']}\n"
            if stats['by_language']:
                result += f"📊 لغات: {', '.join(f'{k}({v})' for k,v in stats['by_language'].items())}"
            return result
        except Exception as e:
            return f"❌ خطأ: {e}"

    def clear_rag():
        global vector_store
        vector_store = Chroma(
            collection_name="erp_sap_codebase",
            embedding_function=embedding_model,
            persist_directory="./chroma_erp_db"
        )
        vector_store.persist()
        return "✅ تم مسح الذاكرة المتجهة"

    def show_stats():
        count = vector_store._collection.count()
        gpu_info = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
        vram = f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB" if torch.cuda.is_available() else "N/A"
        return (
            f"📌 مقاطع مفهرسة: {count}\n"
            f"🤖 النموذج: {MODEL_ID.split('/')[-1]}\n"
            f"🖥️ GPU: {gpu_info}\n"
            f"📦 VRAM: {vram}"
        )

    # ربط الأحداث
    msg.submit(respond, [msg, chatbot], [msg, chatbot])
    send_btn.click(respond, [msg, chatbot], [msg, chatbot])
    clear.click(lambda: ([], ""), None, [chatbot, msg])
    file_upload.upload(upload_project_handler, file_upload, upload_status)
    btn_clear_rag.click(clear_rag, None, upload_status)
    btn_stats.click(show_stats, None, stats_display)


# -------------------- الجزء 7: تشغيل الخادمات --------------------
from pyngrok import ngrok

def run_uvicorn():
    """تشغيل FastAPI في thread منفصل."""
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning")

# تشغيل FastAPI
api_thread = threading.Thread(target=run_uvicorn, daemon=True)
api_thread.start()
time.sleep(3)

# إعداد Ngrok — قراءة التوكن من متغير بيئة (أمان)
NGROK_AUTH_TOKEN = os.environ.get("NGROK_AUTH_TOKEN", "")
if NGROK_AUTH_TOKEN:
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
    public_url_api = ngrok.connect(8000, "http")
    public_url_gradio = ngrok.connect(7860, "http")
    print("\n" + "=" * 60)
    print(f"🔗 OpenAI-Compatible API: {public_url_api}/v1")
    print(f"🌐 Gradio UI: {public_url_gradio}")
    print("=" * 60)
    print("\n📋 لاستخدام مع Cursor/VS Code:")
    print(f'   API Base URL: {public_url_api}/v1')
    print(f'   Model: {MODEL_ID}')
else:
    print("\n" + "=" * 60)
    print("⚠️ NGROK_AUTH_TOKEN غير محدد — التشغيل محلي فقط")
    print(f"🔗 Local API: http://localhost:8000/v1")
    print(f"🌐 Local Gradio: http://localhost:7860")
    print("=" * 60)
    print('\n💡 لتفعيل Ngrok: os.environ["NGROK_AUTH_TOKEN"] = "your_token"')

print(f"\n🤖 النموذج: {MODEL_ID}")
print(f"📦 GPU Tier: {gpu_tier}")
print(f"📚 الوحدات المدعومة: {len(ERP_MODULES)} وحدة ERP")
print(f"🗂️ الامتدادات المدعومة: {len(ERP_FILE_EXTENSIONS)} نوع ملف")
print()

# تشغيل Gradio
demo.launch(server_name="0.0.0.0", server_port=7860, share=False, debug=True)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.2/129.2 kB 6.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.3/31.3 MB 62.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.6/133.6 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 99.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 93.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 596.4/596.4 kB 34.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 78.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

/tmp/ipykernel_2100/3726297871.py:222: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vector_store = Chroma(
/tmp/ipykernel_2100/3726297871.py:966: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(


TypeError: Chatbot.__init__() got an unexpected keyword argument 'show_copy_button'